### Import Dependencies

In [ ]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings

### Download an example reference data point from LangSmith

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

In [ ]:
ls_client = Client()

In [ ]:
dataset = ls_client.read_dataset(dataset_name="rag-evaluation-dataset")

In [ ]:
dataset

In [ ]:
examples = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))
first_example_with_multiple_ref_context_ids = next(
    (
        example
        for example in examples
        if example.outputs and len(example.outputs["reference_context_ids"]) > 1
    ),
    None
)

if not first_example_with_multiple_ref_context_ids:
    raise ValueError("No example with multiple chunks found")

first_example_with_multiple_ref_context_ids.outputs

In [ ]:

if not first_example_with_multiple_ref_context_ids:
    raise ValueError("No example with multiple chunks found")

first_example_with_multiple_ref_context_ids.inputs

In [ ]:

if not first_example_with_multiple_ref_context_ids:
    raise ValueError("No example with multiple chunks found")

reference_input = first_example_with_multiple_ref_context_ids.inputs
reference_output = first_example_with_multiple_ref_context_ids.outputs

### RAG Pipeline

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

embedding_model = "text-embedding-3-small"

def get_embedding(text, model=embedding_model):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01", query=query_embedding, limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint")
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


def process_context(context):
    formatted_context = ""

    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt


answer_gen_model = "gpt-5.4-nano"


def generate_answer(prompt):
    response = openai.chat.completions.create(
        model=answer_gen_model,
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning_effort="none",
    )

    return response.choices[0].message.content


def rag_pipeline(question, topk_k=5):

    retrieved_context = retrieve_data(question, k=topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    if not answer:
        raise ValueError("LLM returned no content")

    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
    }

    return final_answer

In [ ]:
rag_pipeline("Can I get a charger?")

### RAGAS Metrics

In [ ]:
from ragas import SingleTurnSample
from ragas.metrics.collections import Faithfulness, AnswerRelevancy
from ragas.metrics import (
    IDBasedContextPrecision,
    IDBasedContextRecall,
    ResponseRelevancy,
)
from ragas.embeddings.base import embedding_factory

from ragas.llms import llm_factory
from openai import AsyncOpenAI

In [ ]:
client = AsyncOpenAI()
ragas_llm = llm_factory(
    "gpt-5.4-mini",
    client=client,
    max_completion_tokens=4096,
    temperature=1.0,
)
ragas_embeddings = RagasOpenAIEmbeddings(model="text-embedding-3-small", client=client)

# gpt-5.4-mini uses decimal versioning; ragas only auto-maps max_tokens for gpt-5, gpt-6, etc.
ragas_llm.model_args.pop("max_tokens", None)  # pyright: ignore[reportAttributeAccessIssue]
ragas_llm.model_args.pop("top_p", None)  # pyright: ignore[reportAttributeAccessIssue]

In [ ]:
reference_input

In [ ]:
reference_output

In [ ]:
if not reference_input:
  raise ValueError("No reference input found")

real_rag_pipeline_output = rag_pipeline(reference_input["question"])


real_rag_pipeline_output

In [ ]:
async def ragas_context_precision_id_based(run, example):
  sample = SingleTurnSample(
    retrieved_context_ids=run["retrieved_context_ids"],
    reference_context_ids=example["reference_context_ids"],
  )

  scorer = IDBasedContextPrecision()

  return await scorer.single_turn_ascore(sample)
  

In [ ]:
rating = await ragas_context_precision_id_based(real_rag_pipeline_output, reference_output)

In [ ]:
rating

In [ ]:
async def ragas_context_recall_id_based(run, example):
  sample = SingleTurnSample(
    retrieved_context_ids=run["retrieved_context_ids"],
    reference_context_ids=example["reference_context_ids"],
  )

  scorer = IDBasedContextRecall()
  return await scorer.single_turn_ascore(sample)

In [ ]:
recall_score = await ragas_context_recall_id_based(real_rag_pipeline_output, reference_output)

In [ ]:
recall_score

In [ ]:
async def ragas_faithfulness(run):
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

In [ ]:
faithfulness_score = await ragas_faithfulness(real_rag_pipeline_output)

In [ ]:
faithfulness_score.value

In [ ]:


async def ragas_relevancy(run):
    scorer = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    return await scorer.ascore(
        user_input=run["question"],
        response=run["answer"]
    )

In [ ]:
relevancy_score = await ragas_relevancy(real_rag_pipeline_output)

In [ ]:
relevancy_score.value